# map_filter_reduce
**Prerequisites:** function_basics, parameters_and_arguments, return_values, lambda_functions, higher_order_functions

**Outcome:** After this notebook you will know exactly what map, filter, and reduce do under the hood, when they are the right tool, and when a list comprehension or explicit loop is cleaner.

## The Problem

In [ ]:
# transforming, filtering, and aggregating records
# with manual loops — verbose, hard to chain

records = [
    {"id": "job_1", "duration_ms": 120, "status": "done"},
    {"id": "job_2", "duration_ms": 300, "status": "failed"},
    {"id": "job_3", "duration_ms": 80,  "status": "done"},
    {"id": "job_4", "duration_ms": 450, "status": "done"},
]

# step 1: keep only done jobs
done = []
for r in records:
    if r["status"] == "done":
        done.append(r)

# step 2: extract duration
durations = []
for r in done:
    durations.append(r["duration_ms"])

# step 3: sum
total = 0
for d in durations:
    total += d

print(total)   # 650

## The Concept

`map`, `filter`, and `reduce` are three fundamental higher-order functions that capture the most common patterns for working with collections. `map` applies a function to every element and produces a transformed sequence. `filter` applies a predicate function to every element and produces only the elements for which it returns True. `reduce` applies a function cumulatively to elements left to right, collapsing the sequence to a single value. In Python, `map` and `filter` are built-ins that return lazy iterators. `reduce` lives in `functools`. All three accept a function and an iterable.

## Minimal Example

In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5]

doubled  = list(map(lambda x: x * 2, numbers))
evens    = list(filter(lambda x: x % 2 == 0, numbers))
total    = reduce(lambda acc, x: acc + x, numbers)

print(doubled)   # [2, 4, 6, 8, 10]
print(evens)     # [2, 4]
print(total)     # 15

## How Python Does It

`map` and `filter` return iterator objects — they do not build the full result list in memory. Each element is produced lazily only when requested. This means they use O(1) memory regardless of input size. `reduce` is eager — it processes the entire sequence immediately and returns a single value. Wrapping `map` or `filter` in `list()` forces full evaluation and produces a list in memory. If you only need to iterate once, skip the `list()` wrap and let the iterator be consumed lazily.

In [ ]:
# map and filter are lazy iterators
numbers = [1, 2, 3, 4, 5]

mapped   = map(lambda x: x * 2, numbers)
filtered = filter(lambda x: x % 2 == 0, numbers)

print(type(mapped))    # <class 'map'>
print(type(filtered))  # <class 'filter'>

# nothing computed yet — pull values with next() or iterate
print(next(mapped))    # 2
print(next(mapped))    # 4

# force full evaluation
print(list(filter(lambda x: x % 2 == 0, numbers)))   # [2, 4]

## Building Up

In [ ]:
# map — transform every element
records = [
    {"id": "job_1", "duration_ms": 120},
    {"id": "job_2", "duration_ms": 300},
    {"id": "job_3", "duration_ms": 80},
]

# extract one field from every record
ids       = list(map(lambda r: r["id"],          records))
durations = list(map(lambda r: r["duration_ms"], records))

print(ids)
print(durations)

In [ ]:
# map with a named function — cleaner than lambda for non-trivial transforms
def to_seconds(record):
    return {**record, "duration_s": record["duration_ms"] / 1000}

records = [
    {"id": "job_1", "duration_ms": 1200},
    {"id": "job_2", "duration_ms": 3000},
]

converted = list(map(to_seconds, records))
print(converted)

In [ ]:
# map over multiple iterables — function receives one element from each
ids      = ["job_1", "job_2", "job_3"]
statuses = ["done",  "failed", "pending"]

pairs = list(map(lambda i, s: {"id": i, "status": s}, ids, statuses))
print(pairs)

In [ ]:
# filter — keep elements where predicate returns True
records = [
    {"id": "job_1", "status": "done",    "retries": 0},
    {"id": "job_2", "status": "failed",  "retries": 3},
    {"id": "job_3", "status": "done",    "retries": 1},
    {"id": "job_4", "status": "pending", "retries": 0},
]

done      = list(filter(lambda r: r["status"] == "done",  records))
retried   = list(filter(lambda r: r["retries"] > 0,       records))
actionable = list(filter(lambda r: r["status"] != "done", records))

print("done     :", [r["id"] for r in done])
print("retried  :", [r["id"] for r in retried])
print("actionable:", [r["id"] for r in actionable])

In [ ]:
# filter with None — removes all falsy values
values = [0, 1, "", "api", None, False, True, [], [1, 2]]
truthy = list(filter(None, values))
print(truthy)   # [1, 'api', True, [1, 2]]

In [ ]:
# reduce — collapse a sequence to a single value
from functools import reduce

latencies = [120, 300, 80, 450, 95]

total   = reduce(lambda acc, x: acc + x, latencies)
maximum = reduce(lambda acc, x: acc if acc > x else x, latencies)

print(f"total  : {total}")
print(f"maximum: {maximum}")

In [ ]:
# reduce with initial value — safe for empty sequences
from functools import reduce

def merge_configs(configs):
    return reduce(lambda acc, cfg: {**acc, **cfg}, configs, {})

configs = [
    {"host": "localhost"},
    {"port": 5432},
    {"timeout": 30, "host": "remotehost"},   # later values override earlier
]

print(merge_configs(configs))
print(merge_configs([]))   # empty — returns initial value {}

In [ ]:
# chaining map and filter — lazy, no intermediate lists
records = [
    {"id": "job_1", "duration_ms": 120, "status": "done"},
    {"id": "job_2", "duration_ms": 300, "status": "failed"},
    {"id": "job_3", "duration_ms": 80,  "status": "done"},
    {"id": "job_4", "duration_ms": 450, "status": "done"},
]

# filter then map — no intermediate list allocated
done_durations = map(
    lambda r: r["duration_ms"],
    filter(lambda r: r["status"] == "done", records)
)

print(sum(done_durations))   # 650

## Where It Breaks

In [ ]:
# map and filter iterators can only be consumed once
numbers = [1, 2, 3, 4, 5]
doubled = map(lambda x: x * 2, numbers)

print(list(doubled))   # [2, 4, 6, 8, 10]
print(list(doubled))   # [] — iterator exhausted

In [ ]:
# reduce on an empty sequence without initial value raises TypeError
from functools import reduce

try:
    result = reduce(lambda acc, x: acc + x, [])
except TypeError as e:
    print(e)

# always provide initial value when the sequence may be empty
result = reduce(lambda acc, x: acc + x, [], 0)
print(result)   # 0

## The Fix

In [ ]:
# fix: wrap in list() if you need to iterate more than once
numbers = [1, 2, 3, 4, 5]
doubled = list(map(lambda x: x * 2, numbers))   # materialised — reusable

print(doubled)
print(doubled)   # still works

In [ ]:
# solving the original problem cleanly
from functools import reduce

records = [
    {"id": "job_1", "duration_ms": 120, "status": "done"},
    {"id": "job_2", "duration_ms": 300, "status": "failed"},
    {"id": "job_3", "duration_ms": 80,  "status": "done"},
    {"id": "job_4", "duration_ms": 450, "status": "done"},
]

total = reduce(
    lambda acc, d: acc + d,
    map(
        lambda r: r["duration_ms"],
        filter(lambda r: r["status"] == "done", records)
    ),
    0
)
print(total)   # 650

# or even simpler with sum()
total = sum(r["duration_ms"] for r in records if r["status"] == "done")
print(total)   # 650

## In a Real System

In [ ]:
from functools import reduce

# a metrics aggregator that uses map, filter, and reduce
# to produce a region-level summary from raw job records

raw_jobs = [
    {"id": "job_1", "region": "us-east", "status": "done",   "duration_ms": 120},
    {"id": "job_2", "region": "eu-west", "status": "done",   "duration_ms": 300},
    {"id": "job_3", "region": "us-east", "status": "failed", "duration_ms": 80},
    {"id": "job_4", "region": "us-east", "status": "done",   "duration_ms": 450},
    {"id": "job_5", "region": "eu-west", "status": "done",   "duration_ms": 200},
]

def summarise_region(jobs, region):
    regional = list(filter(lambda j: j["region"] == region, jobs))
    done     = list(filter(lambda j: j["status"] == "done",  regional))
    durations = list(map(lambda j: j["duration_ms"], done))
    total_ms  = reduce(lambda acc, d: acc + d, durations, 0)
    return {
        "region":   region,
        "total":    len(regional),
        "done":     len(done),
        "total_ms": total_ms,
        "avg_ms":   round(total_ms / len(done), 2) if done else 0,
    }

for region in ["us-east", "eu-west"]:
    print(summarise_region(raw_jobs, region))

## Performance

`map` and `filter` are lazy — O(1) memory regardless of input size. This is their main advantage over list comprehensions when the collection is large and you only need to iterate once. `reduce` is O(n) time and O(1) memory. For simple aggregations like sum, max, min, any, all — use the built-in functions directly rather than reduce. They are implemented in C, faster, and more readable. Use `reduce` only when no built-in covers your aggregation pattern.

## Anti-Pattern

In [ ]:
from functools import reduce

# anti-pattern: using reduce when a built-in does the job
numbers = [1, 2, 3, 4, 5]

# bad — reduce for sum
total = reduce(lambda acc, x: acc + x, numbers)

# good — built-in is faster and clearer
total = sum(numbers)
print(total)

# bad — reduce for max
maximum = reduce(lambda acc, x: acc if acc > x else x, numbers)

# good — built-in
maximum = max(numbers)
print(maximum)

# anti-pattern: map/filter where a comprehension is more readable
ids = list(map(lambda r: r["id"], [{"id": "job_1"}, {"id": "job_2"}]))

# better — comprehension
ids = [r["id"] for r in [{"id": "job_1"}, {"id": "job_2"}]]
print(ids)

## Interview Signals

- What does `map` return in Python 3 and why is that different from Python 2?
- What happens if you call `list()` on a `map` object twice?
- Why does `reduce` raise `TypeError` on an empty sequence without an initial value?
- When would you prefer `map`/`filter` over a list comprehension?

## Exercise

In [ ]:
from functools import reduce

def analyse_jobs(jobs):
    """
    Takes a list of job dicts with keys: id, status, duration_ms, region.
    Returns a dict with:
        'failed_ids'     : sorted list of ids where status == 'failed'
        'done_avg_ms'    : average duration_ms of done jobs, rounded to 2dp
                           0 if no done jobs
        'region_counts'  : dict mapping each region to total job count
                           built using reduce — no for loops allowed

    Constraints:
        - use filter() for failed_ids
        - use map() and filter() for done_avg_ms
        - use reduce() for region_counts
        - no explicit for loops anywhere in this function

    Bug: the function body is missing — implement it.
    """
    pass


jobs = [
    {"id": "job_1", "status": "done",   "duration_ms": 100, "region": "us-east"},
    {"id": "job_2", "status": "failed", "duration_ms": 200, "region": "eu-west"},
    {"id": "job_3", "status": "done",   "duration_ms": 300, "region": "us-east"},
    {"id": "job_4", "status": "failed", "duration_ms": 150, "region": "eu-west"},
    {"id": "job_5", "status": "done",   "duration_ms": 200, "region": "ap-south"},
]

result = analyse_jobs(jobs)

assert result["failed_ids"]              == ["job_2", "job_4"],  f"got {result['failed_ids']}"
assert result["done_avg_ms"]             == 200.0,               f"got {result['done_avg_ms']}"
assert result["region_counts"]["us-east"]  == 2,                f"got {result['region_counts']}"
assert result["region_counts"]["eu-west"]  == 2,                f"got {result['region_counts']}"
assert result["region_counts"]["ap-south"] == 1,                f"got {result['region_counts']}"

assert analyse_jobs([])["failed_ids"]   == []
assert analyse_jobs([])["done_avg_ms"]  == 0
assert analyse_jobs([])["region_counts"] == {}

print("all assertions passed")